In [4]:
from qiskit import QuantumCircuit

state_prep = QuantumCircuit(4)
# state_prep.h(0)
# state_prep.h(1)
# state_prep.h(2)
# state_prep.s(2)
# state_prep.h(3)
# state_prep.s(3)
state_prep.h(3)
state_prep.h(0)
state_prep.s(0)

state_prep.draw()

┌───┐┌───┐
q_0: ┤ H ├┤ S ├
     └───┘└───┘
q_1: ──────────
               
q_2: ──────────
     ┌───┐     
q_3: ┤ H ├─────
     └───┘

In [5]:
from utils.postselection import paulis_diagonalization_circuit
from qiskit.quantum_info import Pauli, PauliList, SparsePauliOp

pops = PauliList([Pauli("ZZZZ"), Pauli("XXYY"), Pauli("YYXX")])
obsops = PauliList([Pauli("IYXI"), Pauli("YIIX"), Pauli("XZZY")])

# pops = PauliList([Pauli("ZZZZ"), Pauli("YYXX"), Pauli("XXYY")])
# obsops = PauliList([Pauli("IXYI"), Pauli("XIIY"), Pauli("YZZX")])

# circ, R_inv = paulis_diagonalization_circuit(obsops + pops)
basis_change_circ, R_inv, phases = paulis_diagonalization_circuit(pops + obsops)
print(R_inv)
print(phases)
basis_change_circ.draw()

[[0 1 1 0 1 1]
 [0 1 1 1 0 0]
 [1 1 0 0 0 1]
 [1 1 0 0 0 1]]
[0 0 0 1 1 1]


┌───┐ ░    ┌───┐     
q_0: ┤ S ├─░──■─┤ H ├─────
     ├───┤ ░  │ └───┘┌───┐
q_1: ┤ S ├─░──┼───■──┤ H ├
     ├───┤ ░  │   │  ├───┤
q_2: ┤ H ├─░──┼───■──┤ H ├
     ├───┤ ░  │ ┌───┐└───┘
q_3: ┤ H ├─░──■─┤ H ├─────
     └───┘ ░    └───┘

In [6]:
final_circuit = state_prep.compose(basis_change_circ)
final_circuit_tosim = final_circuit.copy()
final_circuit_tosim.measure_all()
final_circuit.draw()
# final_circuit_tosim.draw()

┌───┐┌───┐┌───┐ ░    ┌───┐     
q_0: ┤ H ├┤ S ├┤ S ├─░──■─┤ H ├─────
     ├───┤└───┘└───┘ ░  │ └───┘┌───┐
q_1: ┤ S ├───────────░──┼───■──┤ H ├
     ├───┤           ░  │   │  ├───┤
q_2: ┤ H ├───────────░──┼───■──┤ H ├
     ├───┤┌───┐      ░  │ ┌───┐└───┘
q_3: ┤ H ├┤ H ├──────░──■─┤ H ├─────
     └───┘└───┘      ░    └───┘

In [7]:
from qiskit_addon_obp import backpropagate
from qiskit_addon_utils.slicing import combine_slices, slice_by_depth, slice_by_gate_types, slice_by_barriers
from qiskit_addon_obp.utils.simplify import OperatorBudget

# slices = slice_by_depth(basis_change_circ, 1)
slices = slice_by_barriers(basis_change_circ)
op_budget = OperatorBudget(max_qwc_groups=2)

observable = SparsePauliOp("ZZIZ")
bp_obs, remaining_slices, metadata = backpropagate(observable, slices, operator_budget=op_budget)
bp_circuit = combine_slices(remaining_slices, include_barriers=True)

bp_obs

SparsePauliOp(['ZZIY'],
              coeffs=[-1.+0.j])

In [8]:
def paulis_from_Rinv(R_inv):
    paulis = []
    for row in R_inv.T:
        pauli_str = "".join(["Z" if el == 1 else "I" for el in row[::-1]])
        paulis.append(Pauli(pauli_str))
    return PauliList(paulis)

In [9]:
paulis_from_Rinv(R_inv)

PauliList(['ZZII', 'ZZZZ', 'IIZZ', 'IIZI', 'IIIZ', 'ZZIZ'])

In [10]:
from qiskit.quantum_info import Clifford

basis_change_clifford = Clifford(basis_change_circ)
slice_clifford = Clifford(slices[1])

full_backpropagated_observables = [p.evolve(basis_change_clifford, frame="h")[::-1] for p in paulis_from_Rinv(R_inv)]
partially_propagated_observables = [p.evolve(slice_clifford, frame="h")[::-1] for p in paulis_from_Rinv(R_inv)]

In [11]:
full_backpropagated_observables

[Pauli('ZZZZ'),
 Pauli('XXYY'),
 Pauli('YYXX'),
 Pauli('IYXI'),
 Pauli('YIIX'),
 Pauli('XZZY')]

In [12]:
partially_propagated_observables

[Pauli('ZZXX'),
 Pauli('YYYY'),
 Pauli('XXZZ'),
 Pauli('IXZI'),
 Pauli('XIIZ'),
 Pauli('YZXY')]

In [13]:
slices[1].draw()

┌───┐     
q_0: ─■─┤ H ├─────
      │ └───┘┌───┐
q_1: ─┼───■──┤ H ├
      │   │  ├───┤
q_2: ─┼───■──┤ H ├
      │ ┌───┐└───┘
q_3: ─■─┤ H ├─────
        └───┘

In [14]:
from qiskit_aer.primitives import SamplerV2
import numpy as np

qc = QuantumCircuit(4)
qc = qc.compose(state_prep)
qc = qc.compose(slices[0])
# qc.sdg([0, 3])
qc.h([0])

qc.measure_all()

print(qc.draw())

sampler = SamplerV2(options={"backend_options":{"method": "statevector"}})
job = sampler.run([qc], shots=10000)
list(job.result()[0].data.values())[0].get_counts()

        ┌───┐┌───┐┌───┐┌───┐ ░ ┌─┐         
   q_0: ┤ H ├┤ S ├┤ S ├┤ H ├─░─┤M├─────────
        ├───┤└───┘└───┘└───┘ ░ └╥┘┌─┐      
   q_1: ┤ S ├────────────────░──╫─┤M├──────
        ├───┤                ░  ║ └╥┘┌─┐   
   q_2: ┤ H ├────────────────░──╫──╫─┤M├───
        ├───┤┌───┐           ░  ║  ║ └╥┘┌─┐
   q_3: ┤ H ├┤ H ├───────────░──╫──╫──╫─┤M├
        └───┘└───┘           ░  ║  ║  ║ └╥┘
meas: 4/════════════════════════╩══╩══╩══╩═
                                0  1  2  3 


{'0001': 4964, '0101': 5036}

In [15]:
from qiskit_aer.primitives import SamplerV2

shots = 10000

final_circuit = state_prep.compose(basis_change_circ)
final_circuit.measure_all()

print(final_circuit.draw())

sampler = SamplerV2(options={"backend_options": {"method": "statevector"}})
job = sampler.run([final_circuit], shots=10000)
list(job.result()[0].data.values())[0].get_counts()

        ┌───┐┌───┐┌───┐ ░    ┌───┐      ░ ┌─┐         
   q_0: ┤ H ├┤ S ├┤ S ├─░──■─┤ H ├──────░─┤M├─────────
        ├───┤└───┘└───┘ ░  │ └───┘┌───┐ ░ └╥┘┌─┐      
   q_1: ┤ S ├───────────░──┼───■──┤ H ├─░──╫─┤M├──────
        ├───┤           ░  │   │  ├───┤ ░  ║ └╥┘┌─┐   
   q_2: ┤ H ├───────────░──┼───■──┤ H ├─░──╫──╫─┤M├───
        ├───┤┌───┐      ░  │ ┌───┐└───┘ ░  ║  ║ └╥┘┌─┐
   q_3: ┤ H ├┤ H ├──────░──■─┤ H ├──────░──╫──╫──╫─┤M├
        └───┘└───┘      ░    └───┘      ░  ║  ║  ║ └╥┘
meas: 4/═══════════════════════════════════╩══╩══╩══╩═
                                           0  1  2  3 


{'0001': 2481, '1011': 2480, '1001': 2462, '0011': 2577}

Now it is consistent, what happens with my simpler case

In [16]:
pops = PauliList([Pauli("ZZII"), Pauli("IZZZ")])
obsops = PauliList([Pauli("XXXI")])

basis_change_circ, R_inv, phases = paulis_diagonalization_circuit(pops + obsops)
slices = slice_by_barriers(basis_change_circ)
print(basis_change_circ.draw())
print(R_inv)
print(phases)
print(slices[1])

           ░         ┌───┐
q_0: ──────░───■───■─┤ H ├
     ┌───┐ ░   │   │ ├───┤
q_1: ┤ H ├─░───■───┼─┤ H ├
     ├───┤ ░       │ ├───┤
q_2: ┤ H ├─░───────■─┤ H ├
     ├───┤ ░ ┌───┐   └───┘
q_3: ┤ H ├─░─┤ H ├────────
     └───┘ ░ └───┘        
[[0 0 1]
 [1 1 0]
 [0 1 0]
 [0 1 0]]
[0 0 0]
             ┌───┐
q_0: ──■───■─┤ H ├
       │   │ ├───┤
q_1: ──■───┼─┤ H ├
           │ ├───┤
q_2: ──────■─┤ H ├
     ┌───┐   └───┘
q_3: ┤ H ├────────
     └───┘        


In [17]:
pops = PauliList([Pauli("ZZIIIIIII"), Pauli("IZZZIIIII"), Pauli("IIIZZZIII"), Pauli("IIIIIZZZI"), Pauli("IIIIIIIZZ")])

different_paulis = set("".join([str(op) for op in pops]))
different_paulis

{'I', 'Z'}

In [18]:
pops = PauliList([Pauli("ZZIIIIIII"), Pauli("IZZZIIIII"), Pauli("IIIZZZIII"), Pauli("IIIIIZZZI"), Pauli("IIIIIIIZZ")])
obsops = PauliList([Pauli("XXXIIIIII"), Pauli("IIXXXIIII"), Pauli("IIIIXXXII"), Pauli("IIIIIIXXX")])

from qiskit_addon_utils.slicing import slice_by_barriers

basis_change_circ, R_inv, phases = paulis_diagonalization_circuit(pops + obsops)
slices = slice_by_barriers(basis_change_circ)
print(basis_change_circ.draw())
print(R_inv)
print(phases)
print(slices[1])

LinAlgError: Singular matrix

In [20]:
pops = PauliList([Pauli("ZZIIIIIII"), Pauli("IZZZIIIII"), Pauli("IIIZZZIII"), Pauli("IIIIIZZZI"), Pauli("IIIIIIIZZ")])
obsops = PauliList([Pauli("XXXIIIIII"), Pauli("IIXXXIIII")])

from qiskit_addon_utils.slicing import slice_by_barriers

basis_change_circ, R_inv, phases = paulis_diagonalization_circuit(pops + obsops)
slices = slice_by_barriers(basis_change_circ)
print(basis_change_circ.draw())
print(R_inv)
print(phases)
print(slices[1])

LinAlgError: Singular matrix

In [19]:
from qiskit.quantum_info import Clifford

basis_change_clifford = Clifford(basis_change_circ)
slice_clifford = Clifford(slices[1])

full_backpropagated_observables = [p.evolve(basis_change_clifford, frame="h")[::-1] for p in paulis_from_Rinv(R_inv)]
partially_propagated_observables = [p.evolve(slice_clifford, frame="h")[::-1] for p in paulis_from_Rinv(R_inv)]

print(full_backpropagated_observables)
print(partially_propagated_observables)

[Pauli('ZZII'), Pauli('IZZZ'), Pauli('XXXI')]
[Pauli('ZXII'), Pauli('IXXX'), Pauli('XZZI')]


In [20]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(4)
qc.cz(0, 3)

qc.draw()

q_0: ─■─
      │ 
q_1: ─┼─
      │ 
q_2: ─┼─
      │ 
q_3: ─■─

In [21]:
from qiskit_ibm_runtime import QiskitRuntimeService
from cobos.tokens import ibm_token, group_instance
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

service = QiskitRuntimeService(channel="ibm_quantum", token=ibm_token, instance=group_instance)
backend = service.backend("ibm_fez")

In [22]:
pm = generate_preset_pass_manager(optimization_level=0, backend=backend, initial_layout=[0, 1, 2, 3])
pc = pm.run(qc)
pc.measure_active()
pc.draw(idle_wires=False)

global phase: π
           ┌────┐   ┌────┐   ┌────┐       ░ ┌─┐         
  q_0 -> 0 ┤ √X ├─■─┤ √X ├─■─┤ √X ├─■─────░─┤M├─────────
           ├────┤ │ ├────┤ │ ├────┤ │     ░ └╥┘┌─┐      
  q_1 -> 1 ┤ √X ├─■─┤ √X ├─■─┤ √X ├─■──■──░──╫─┤M├──────
           ├────┤   ├────┤   ├────┤    │  ░  ║ └╥┘┌─┐   
  q_2 -> 2 ┤ √X ├─■─┤ √X ├─■─┤ √X ├─■──■──░──╫──╫─┤M├───
           ├────┤ │ ├────┤ │ ├────┤ │     ░  ║  ║ └╥┘┌─┐
  q_3 -> 3 ┤ √X ├─■─┤ √X ├─■─┤ √X ├─■─────░──╫──╫──╫─┤M├
           └────┘   └────┘   └────┘       ░  ║  ║  ║ └╥┘
measure: 4/══════════════════════════════════╩══╩══╩══╩═
                                             0  1  2  3